In [42]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pathlib import Path
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, GradientBoostingClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import mean_squared_error, r2_score

def load(path):
    p = Path(path)
    return pd.read_csv(p) if p.exists() else None

train  = load("execution_module/data/train.csv")
credit = load("execution_module/data/credit_risk_dataset.csv")
house  = load("execution_module/data/housing copy.csv")



In [43]:
for name, df in [("Train", train), ("Credit Risk", credit), ("House Prices", house)]:
    print(f"\n{'='*40}")
    print(f"Dataset: {name}")
    print(f"Shape: {df.shape}") 
    print(f"Columns: {list(df.columns)}")
    print(f"\nMissing values:\n{df.isnull().sum()}")
    print(f"\nFirst 3 rows:\n{df.head(3)}")


Dataset: Train
Shape: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

First 3 rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0

In [44]:
# makes a copy of the dataframe, drops irrelevant columns, separates features and target, fills missing values, and encodes categorical variables.

def preprocess(df, target_col, dataset_name):
    df = df.copy()
    
    cols_to_drop = {
    "train": ["PassengerId", "Name", "Ticket", "Cabin"],
    "credit": [],
    "house": []
}.get(dataset_name, [])
    
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    y = df[target_col]
    X = df.drop(columns=[target_col])
    
# fill missing values: numeric with median, categorical with mode
    for col in X.select_dtypes(include=np.number).columns:
        X[col] = X[col].fillna(X[col].median())
        
    for col in X.select_dtypes(include="object").columns:
        X[col] = X[col].fillna(X[col].mode()[0])
        
   # encode categorical variables 
    le = LabelEncoder()
    for col in X.select_dtypes(include="object").columns:
        X[col] = le.fit_transform(X[col])
        
    return X, y
    


In [45]:
X, y = preprocess(train, "Survived", "train")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFirst 3 rows of X:")
print(X.head(3))

X shape: (891, 7)
y shape: (891,)

First 3 rows of X:
   Pclass  Sex   Age  SibSp  Parch     Fare  Embarked
0       3    1  22.0      1      0   7.2500         2
1       1    0  38.0      1      0  71.2833         0
2       3    0  26.0      0      0   7.9250         2


C:\Users\maste\AppData\Local\Temp\ipykernel_48740\1081079555.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include="object").columns:
C:\Users\maste\AppData\Local\Temp\ipykernel_48740\1081079555.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide

In [46]:
def split_data(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print(f"Training size: {X_train.shape[0]} rows")
    print(f"Testing size: {X_test.shape[0]} rows")
    return X_train, X_test, y_train, y_test

In [47]:
X_train, X_test, y_train, y_test = split_data(X, y)

Training size: 712 rows
Testing size: 179 rows


In [48]:
def train_model(model_name, X_train, y_train):
    models ={
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
        "XGBoost": XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
    }
    
    if model_name not in models:
        print(f"Model '{model_name}' not found.")
        return None
    
    model = models[model_name]
    model.fit(X_train, y_train)
    print(f"{model_name} trained successfully.")
    return model


In [49]:
model = train_model("Random Forest", X_train, y_train)


Random Forest trained successfully.


In [50]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    
    results = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
        "f1": f1_score(y_test, predictions),
        "roc_auc": roc_auc_score(y_test, probabilities)
    }
    
    for metric, value in results.items():
        print(f"{metric}: {round(value, 4)}")
        
    return results

In [51]:
results = evaluate_model(model, X_test, y_test)

accuracy: 0.8212
precision: 0.8088
recall: 0.7432
f1: 0.7746
roc_auc: 0.8988


In [52]:
X_credit, y_credit = preprocess(credit, "loan_status", "credit")
X_train_c, X_test_c, y_train_c, y_test_c = split_data(X_credit, y_credit)

Training size: 26064 rows
Testing size: 6517 rows


C:\Users\maste\AppData\Local\Temp\ipykernel_48740\1081079555.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include="object").columns:
C:\Users\maste\AppData\Local\Temp\ipykernel_48740\1081079555.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide

In [53]:
model_credit = train_model("Random Forest", X_train_c, y_train_c)

Random Forest trained successfully.


In [54]:
results_credit = evaluate_model(model_credit, X_test_c, y_test_c)

accuracy: 0.9297
precision: 0.9643
recall: 0.7093
f1: 0.8174
roc_auc: 0.9349


In [55]:
def train_model_regression(model_name, X_train, y_train):
    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
        "XGBoost": XGBRegressor(n_estimators=100, random_state=42)
    }
    
    if model_name not in models:
        print(f"Model '{model_name}' not found.")
        return None
    
    model = models[model_name]
    model.fit(X_train, y_train)
    print(f"{model_name} trained successfully!")
    return model

In [56]:
def evaluate_model_regression(model, X_test, y_test):
    predictions = model.predict(X_test)
    
    results = {
         "rmse": round(np.sqrt(mean_squared_error(y_test, predictions)), 4),
        "r2": round(r2_score(y_test, predictions), 4)
    }
    
    for metric, value in results.items():
        print(f"{metric}: {value}")
    
    return results

In [57]:
X_house, y_house = preprocess(house, "median_house_value", "house")
X_train_h, X_test_h, y_train_h, y_test_h = split_data(X_house, y_house)

Training size: 16512 rows
Testing size: 4128 rows


C:\Users\maste\AppData\Local\Temp\ipykernel_48740\1081079555.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include="object").columns:
C:\Users\maste\AppData\Local\Temp\ipykernel_48740\1081079555.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide

In [58]:
model_house = train_model_regression("Random Forest", X_train_h, y_train_h)

Random Forest trained successfully!


In [59]:
results_house = evaluate_model_regression(model_house, X_test_h, y_test_h)

rmse: 50231.0768
r2: 0.8075
